# Computational scalability benchmark

Finite complete-model libraries are evaluated as source count, evidence-alphabet size and library size vary. The benchmark records dynamic-programming states, solver time, peak resident memory, Static double-oracle/LP diagnostics, Rectangular belief-set growth and policy-reconstruction/evaluation costs.

Runtime, memory and state counts are the primary endpoints. Risk values and hierarchy checks are correctness diagnostics. Methods run in isolated child processes by default. Time and state limits are retained as `timeout` and `state_budget_exceeded` observations; they must not be treated as completed runtimes. Solver and later policy-evaluation statuses remain separate.

Outputs are written under `revision_issue4/scalability_benchmark`.


In [ ]:
from __future__ import annotations

import gc
import itertools
import json
import math
import multiprocessing as mp
import os
import platform
import signal
import sys
import threading
import time
import traceback
import tracemalloc
import warnings
from contextlib import contextmanager
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import linprog

try:
    import psutil
except Exception:
    psutil = None



# Configuration


BASE_SEED = 20261004

# The three sweeps are deliberately separated. This avoids a very expensive
# full Cartesian product while still identifying the effect of each axis.
SOURCE_VALUES = (3, 4, 5, 6, 7, 8)
ALPHABET_VALUES = (2, 3)
MODEL_VALUES = (1, 3, 5, 8)

RUN_HIGH_SOURCE_CASES = True
RUN_ALPHABET_SWEEP = True
RUN_MODEL_SWEEP = True

PRIOR_Y1 = 0.55
ABSTENTION_COST = 0.20
STATIC_TOLERANCE = 1e-9
NUMERICAL_TOLERANCE = 1e-8

# Safety controls. Set MAX_MEMO_STATES=None only after checking that the
# JupyterHub kernel has enough RAM for the requested exact horizon.
MAX_MEMO_STATES = 1_500_000
MAX_DO_ITERATIONS = 500
MAX_CASE_SECONDS = 180.0
MAX_PEAK_MEMORY_MB = 8192.0

# Small cases are repeated to reduce timing noise. Large cases are run once.
SMALL_CASE_REPEAT_LIMIT_M = 4
SMALL_CASE_REPEATS = 3

# RSS sampling is the primary memory metric. Python allocation tracing is
# optional because it can materially slow large NumPy/dictionary workloads.
MEASURE_PYTHON_ALLOCATIONS = False
RSS_POLL_SECONDS = 0.005

# A fresh child process is the default measurement unit. In a long-lived
# Jupyter kernel, Python/NumPy allocators may retain freed pages; measuring all
# cases in the kernel would therefore make later RSS deltas incomparable.
# JupyterHub's usual Linux runtime provides ``fork``. If the requested start
# method is unavailable, the benchmark stops with an explicit diagnostic
# rather than silently reporting contaminated memory numbers.
ISOLATE_METHOD_RUNS = True
MP_START_METHOD = "fork"
PROCESS_GRACE_SECONDS = 20.0

OUTPUT_DIR = Path("revision_issue4") / "scalability_benchmark"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METHODS = ("NominalDP", "HCR", "Rectangular", "StaticDO")
METRIC_NAMES = ("risk", "wrong", "abstain", "coverage", "cost", "queries")

MAX_BENCHMARK_SOURCES = max(SOURCE_VALUES)
MAX_BENCHMARK_MODELS = max(MODEL_VALUES)

print("Configuration loaded.")
print("Output directory:", OUTPUT_DIR.resolve())
print("Source sweep:", SOURCE_VALUES)
print("Alphabet sweep:", ALPHABET_VALUES)
print("Model sweep:", MODEL_VALUES)

## Benchmark conventions

The finite model library has shape

\[
\texttt{channels}[k,i,y,e],
\]

where (k=1,ldots,K) indexes complete source models, (i=1,ldots,m) indexes sources, (yin\{0,1\}) indexes the latent class, and (ein\{0,ldots,A-1}) indexes encoded evidence.

The source family is nested for the (m)- and (K)-sweeps: increasing (m) adds sources without regenerating the earlier sources, and increasing (K) adds complete models without changing the preceding models. This makes runtime/state comparisons interpretable. The (A)-sweep uses a separately seeded categorical family because changing the alphabet changes the channel dimension itself.

The exact algorithms use no heuristic pruning. The only default restrictions are the explicit safety timeout and memo-state budget, both of which are reported in the output.

In [ ]:

# Exceptions, serialization, and deterministic benchmark generation



class StateBudgetExceeded(RuntimeError):
    pass


class BenchmarkTimeout(TimeoutError):
    pass


def _json_default(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"Object of type {type(value).__name__} is not JSON serializable")


def write_json(path: Path, payload):
    path.write_text(
        json.dumps(payload, indent=2, default=_json_default),
        encoding="utf-8",
    )


def _categorical_row(
    alphabet_size: int,
    preferred: int,
    reliability: float,
    probability_floor: float,
) -> np.ndarray:
    if alphabet_size < 2:
        raise ValueError("This benchmark requires alphabet_size >= 2.")
    if not 0.0 < probability_floor < 1.0 / alphabet_size:
        raise ValueError("probability_floor must be in (0, 1/A).")

    row = np.full(
        alphabet_size,
        (1.0 - reliability) / (alphabet_size - 1),
        dtype=float,
    )
    row[int(preferred)] = reliability
    row = np.maximum(row, probability_floor)
    row /= row.sum()
    return row


def build_nested_model_library(
    alphabet_size: int,
    max_sources: int,
    max_models: int,
    seed: int,
    probability_floor: float = 0.02,
) -> tuple[np.ndarray, np.ndarray]:
    """Create a nested finite library of complete source models.

    Returns
    -------
    channels:
        Array with shape (max_models, max_sources, 2, alphabet_size).
    costs:
        Source costs with shape (max_sources,).
    """

    if max_sources < 1 or max_models < 1:
        raise ValueError("max_sources and max_models must be positive.")

    rng = np.random.default_rng(
        np.random.SeedSequence([int(seed), int(alphabet_size), 314159])
    )
    channels = np.zeros(
        (max_models, max_sources, 2, alphabet_size),
        dtype=float,
    )

    for model_index in range(max_models):
        for source_index in range(max_sources):
            reliability = float(rng.uniform(0.58, 0.92))
            preferred_zero = int(rng.integers(0, alphabet_size))
            offset = int(rng.integers(1, alphabet_size))
            preferred_one = (preferred_zero + offset) % alphabet_size

            # Some complete models have a source orientation that differs from
            # the nominal model. The channel remains valid and strictly
            # positive, but the complete model coupling is retained.
            if rng.random() < 0.12:
                preferred_zero, preferred_one = preferred_one, preferred_zero

            channels[model_index, source_index, 0] = _categorical_row(
                alphabet_size,
                preferred_zero,
                reliability,
                probability_floor,
            )
            channels[model_index, source_index, 1] = _categorical_row(
                alphabet_size,
                preferred_one,
                reliability,
                probability_floor,
            )

    cost_rng = np.random.default_rng(
        np.random.SeedSequence([int(seed), int(alphabet_size), 271828])
    )
    costs = cost_rng.uniform(0.003, 0.060, size=max_sources).astype(float)
    return channels, costs


def make_benchmark_problem(
    n_sources: int,
    alphabet_size: int,
    n_models: int,
    seed: int = BASE_SEED,
) -> dict:
    if n_sources < 1:
        raise ValueError("n_sources must be positive.")
    if alphabet_size < 2:
        raise ValueError("alphabet_size must be at least 2.")
    if n_models < 1:
        raise ValueError("n_models must be positive.")

    full_channels, full_costs = build_nested_model_library(
        alphabet_size=alphabet_size,
        max_sources=MAX_BENCHMARK_SOURCES,
        max_models=MAX_BENCHMARK_MODELS,
        seed=seed,
    )
    channels = full_channels[:n_models, :n_sources].copy()
    costs = full_costs[:n_sources].copy()

    if not np.allclose(channels.sum(axis=-1), 1.0, atol=1e-12):
        raise AssertionError("Generated channels are not row-normalized.")
    if np.any(channels <= 0.0):
        raise AssertionError("Generated channels must have positive entries.")

    return {
        "n_sources": int(n_sources),
        "alphabet_size": int(alphabet_size),
        "n_models": int(n_models),
        "prior_y1": float(PRIOR_Y1),
        "abstention_cost": float(ABSTENTION_COST),
        "channels": channels,
        # The first complete model is the nominal model for this computational
        # audit. Robust methods still receive the entire finite library.
        "nominal_channels": channels[0].copy(),
        "costs": costs,
        "seed": int(seed),
    }


def _memo_state_guard(memo: dict, max_states: int | None):
    if max_states is not None and len(memo) >= int(max_states):
        raise StateBudgetExceeded(
            f"memo state budget {max_states} reached before creating a new state"
        )


def _rounded_tuple(values: np.ndarray, decimals: int = 14) -> tuple[float, ...]:
    return tuple(np.round(np.asarray(values, dtype=float), decimals))


def _max_history_depth_from_memo(memo: dict) -> int:
    depths = []
    for key in memo:
        if isinstance(key, tuple) and key:
            first = key[0]
            if isinstance(first, tuple):
                # HCR/Static use tuple((source,evidence), ...); nominal uses
                # an omega tuple. The latter is handled through remaining.
                if first and isinstance(first[0], tuple):
                    depths.append(len(first))
    return int(max(depths, default=0))

## Exact finite-horizon solvers

The next cell implements the finite-horizon recursions used by the audit. The HCR and Static likelihood calculations retain the complete model index (k) throughout each history. Rectangularization instead retains source-wise channel projections and a deduplicated set of possible binary posteriors.

The state budget is an explicit safety mechanism only. When it is reached, the case is marked as incomplete; the result is never silently treated as an exact solution.

In [ ]:

# Nominal DP



def nominal_dp_general(
    prior_y1: float,
    channels: np.ndarray,
    costs: np.ndarray,
    abstention_cost: float,
    max_states: int | None = MAX_MEMO_STATES,
):
    channels = np.asarray(channels, dtype=float)
    costs = np.asarray(costs, dtype=float)
    n_sources, n_hypotheses, alphabet_size = channels.shape
    if n_hypotheses != 2:
        raise ValueError("This implementation assumes binary hypotheses.")
    if len(costs) != n_sources:
        raise ValueError("cost length does not match channel source count.")

    memo = {}

    def recurse(omega: np.ndarray, remaining: tuple[int, ...]):
        key = (tuple(float(value) for value in omega), remaining)
        if key in memo:
            return memo[key]
        _memo_state_guard(memo, max_states)

        total = float(omega.sum())
        if total <= 0.0:
            raise RuntimeError("Nominal DP reached a zero-probability state.")
        posterior_y1 = float(omega[1] / total)

        stop_values = [posterior_y1, 1.0 - posterior_y1, abstention_cost]
        stop_actions = ["answer_0", "answer_1", "abstain"]
        best_value = float(min(stop_values))
        best_action = stop_actions[int(np.argmin(stop_values))]
        continuation_values = {}

        for source_index in remaining:
            next_remaining = tuple(
                j for j in remaining if j != source_index
            )
            continuation = float(costs[source_index])

            for evidence in range(alphabet_size):
                next_omega = omega * channels[source_index, :, evidence]
                evidence_probability = float(next_omega.sum() / total)
                if evidence_probability > 0.0:
                    continuation += evidence_probability * recurse(
                        next_omega,
                        next_remaining,
                    )[0]

            continuation_values[int(source_index)] = float(continuation)
            if continuation < best_value - 1e-12:
                best_value = float(continuation)
                best_action = f"query_{source_index}"

        memo[key] = (
            float(best_value),
            best_action,
            continuation_values,
        )
        return memo[key]

    root = recurse(
        np.array([1.0 - prior_y1, prior_y1], dtype=float),
        tuple(range(n_sources)),
    )
    return root, memo


def nominal_tree_from_memo(
    prior_y1: float,
    nominal_channels: np.ndarray,
    memo: dict,
):
    nominal_channels = np.asarray(nominal_channels, dtype=float)
    n_sources, _, alphabet_size = nominal_channels.shape

    def recurse(omega: np.ndarray, remaining: tuple[int, ...]):
        key = (tuple(float(value) for value in omega), remaining)
        if key not in memo:
            if float(omega.sum()) == 0.0:
                return "abstain", None
            raise KeyError(f"Missing nominal state during reconstruction: {key}")
        action = memo[key][1]
        if not action.startswith("query_"):
            return action, None

        source_index = int(action.split("_")[1])
        next_remaining = tuple(
            j for j in remaining if j != source_index
        )
        children = tuple(
            recurse(
                omega * nominal_channels[source_index, :, evidence],
                next_remaining,
            )
            for evidence in range(alphabet_size)
        )
        return action, children

    return recurse(
        np.array([1.0 - prior_y1, prior_y1], dtype=float),
        tuple(range(n_sources)),
    )



# HCR DP



def hcr_dp_general(
    prior_y1: float,
    scenario_channels: np.ndarray,
    costs: np.ndarray,
    abstention_cost: float,
    max_states: int | None = MAX_MEMO_STATES,
):
    scenario_channels = np.asarray(scenario_channels, dtype=float)
    costs = np.asarray(costs, dtype=float)
    n_models, n_sources, n_hypotheses, alphabet_size = scenario_channels.shape
    if n_hypotheses != 2:
        raise ValueError("This implementation assumes binary hypotheses.")
    if len(costs) != n_sources:
        raise ValueError("cost length does not match channel source count.")

    memo = {}

    def likelihood_matrix(history: tuple[tuple[int, int], ...]) -> np.ndarray:
        likelihood = np.tile(
            np.array([1.0 - prior_y1, prior_y1], dtype=float),
            (n_models, 1),
        )
        for source_index, evidence in history:
            likelihood *= scenario_channels[
                :, source_index, :, evidence
            ]
        return likelihood

    def recurse(
        history: tuple[tuple[int, int], ...],
        remaining: tuple[int, ...],
    ):
        key = (history, remaining)
        if key in memo:
            return memo[key]
        _memo_state_guard(memo, max_states)

        likelihood = likelihood_matrix(history)
        reach_probability = likelihood.sum(axis=1)
        active = reach_probability > 0.0
        if not np.any(active):
            # Every complete model assigns this history zero probability.
            memo[key] = (0.0, "abstain", {}, (0.0, 0.0, float(abstention_cost)))
            return memo[key]

        posterior_y1 = likelihood[active, 1] / reach_probability[active]
        stop_values = [
            float(np.max(posterior_y1)),
            float(np.max(1.0 - posterior_y1)),
            float(abstention_cost),
        ]
        stop_actions = ["answer_0", "answer_1", "abstain"]
        best_value = float(min(stop_values))
        best_action = stop_actions[int(np.argmin(stop_values))]
        continuation_values = {}

        for source_index in remaining:
            next_remaining = tuple(
                j for j in remaining if j != source_index
            )

            child_values = {
                evidence: recurse(
                    history + ((int(source_index), int(evidence)),),
                    next_remaining,
                )[0]
                for evidence in range(alphabet_size)
            }

            scenario_expectations = np.zeros(n_models, dtype=float)
            for evidence in range(alphabet_size):
                numerator = (
                    likelihood
                    * scenario_channels[:, source_index, :, evidence]
                ).sum(axis=1)
                evidence_probability = np.divide(
                    numerator,
                    reach_probability,
                    out=np.zeros_like(numerator),
                    where=active,
                )
                scenario_expectations += (
                    evidence_probability * child_values[evidence]
                )

            continuation_value = float(
                costs[source_index]
                + np.max(scenario_expectations[active])
            )
            continuation_values[int(source_index)] = continuation_value

            if continuation_value < best_value - 1e-12:
                best_value = continuation_value
                best_action = f"query_{source_index}"

        memo[key] = (
            float(best_value),
            best_action,
            continuation_values,
            tuple(float(x) for x in stop_values),
        )
        return memo[key]

    root = recurse(tuple(), tuple(range(n_sources)))
    return root, memo


def hcr_tree_from_memo(memo: dict, n_sources: int, alphabet_size: int):
    def recurse(history, remaining):
        action = memo[(history, remaining)][1]
        if not action.startswith("query_"):
            return action, None
        source_index = int(action.split("_")[1])
        next_remaining = tuple(
            j for j in remaining if j != source_index
        )
        children = tuple(
            recurse(
                history + ((source_index, evidence),),
                next_remaining,
            )
            for evidence in range(alphabet_size)
        )
        return action, children

    return recurse(tuple(), tuple(range(n_sources)))



# Rectangular DP



def _deduplicate_channel_matrices(
    channels: list[np.ndarray],
    decimals: int = 12,
) -> list[np.ndarray]:
    output = []
    seen = set()
    for channel in channels:
        array = np.asarray(channel, dtype=float)
        key = tuple(np.round(array.reshape(-1), decimals))
        if key not in seen:
            seen.add(key)
            output.append(array.copy())
    return output


def _deduplicate_beliefs(values, decimals: int = 12) -> tuple[float, ...]:
    return tuple(sorted(set(round(float(value), decimals) for value in values)))


def rectangular_child_belief_set(
    belief_set: tuple[float, ...],
    source_channels: list[np.ndarray],
    evidence: int,
) -> tuple[float, ...]:
    posteriors = []
    for belief in belief_set:
        for channel in source_channels:
            denominator = (
                (1.0 - belief) * channel[0, evidence]
                + belief * channel[1, evidence]
            )
            if denominator > 0.0:
                posteriors.append(
                    float(belief * channel[1, evidence] / denominator)
                )
    return _deduplicate_beliefs(posteriors)


def rectangular_dp_general(
    prior_y1: float,
    scenario_channels: np.ndarray,
    costs: np.ndarray,
    abstention_cost: float,
    max_states: int | None = MAX_MEMO_STATES,
):
    scenario_channels = np.asarray(scenario_channels, dtype=float)
    costs = np.asarray(costs, dtype=float)
    n_models, n_sources, n_hypotheses, alphabet_size = scenario_channels.shape
    if n_hypotheses != 2:
        raise ValueError("This implementation assumes binary hypotheses.")

    source_channel_sets = [
        _deduplicate_channel_matrices(
            [scenario_channels[k, source_index] for k in range(n_models)]
        )
        for source_index in range(n_sources)
    ]
    memo = {}

    def recurse(
        belief_set: tuple[float, ...],
        remaining: tuple[int, ...],
    ):
        belief_set = _deduplicate_beliefs(belief_set)
        key = (belief_set, remaining)
        if key in memo:
            return memo[key]
        _memo_state_guard(memo, max_states)
        if not belief_set:
            memo[key] = (0.0, "abstain", {}, (0.0, 0.0, float(abstention_cost)))
            return memo[key]

        beliefs = np.asarray(belief_set, dtype=float)
        stop_values = [
            float(np.max(beliefs)),
            float(np.max(1.0 - beliefs)),
            float(abstention_cost),
        ]
        stop_actions = ["answer_0", "answer_1", "abstain"]
        best_value = float(min(stop_values))
        best_action = stop_actions[int(np.argmin(stop_values))]
        continuation_values = {}

        for source_index in remaining:
            next_remaining = tuple(
                j for j in remaining if j != source_index
            )
            child_belief_sets = {
                evidence: rectangular_child_belief_set(
                    belief_set,
                    source_channel_sets[source_index],
                    evidence,
                )
                for evidence in range(alphabet_size)
            }
            child_values = {
                evidence: recurse(
                    child_belief_sets[evidence],
                    next_remaining,
                )[0]
                for evidence in range(alphabet_size)
            }

            worst_expectation = -np.inf
            for belief in beliefs:
                for channel in source_channel_sets[source_index]:
                    expectation = 0.0
                    for evidence in range(alphabet_size):
                        probability = (
                            (1.0 - belief) * channel[0, evidence]
                            + belief * channel[1, evidence]
                        )
                        expectation += probability * child_values[evidence]
                    worst_expectation = max(
                        worst_expectation,
                        expectation,
                    )

            continuation_value = float(
                costs[source_index] + worst_expectation
            )
            continuation_values[int(source_index)] = continuation_value
            if continuation_value < best_value - 1e-12:
                best_value = continuation_value
                best_action = f"query_{source_index}"

        memo[key] = (
            float(best_value),
            best_action,
            continuation_values,
            tuple(float(x) for x in stop_values),
        )
        return memo[key]

    root = recurse((float(prior_y1),), tuple(range(n_sources)))
    return root, memo, source_channel_sets


def rectangular_tree_from_memo(
    prior_y1: float,
    source_channel_sets: list[list[np.ndarray]],
    memo: dict,
    n_sources: int,
    alphabet_size: int,
):
    def recurse(belief_set, remaining):
        belief_set = _deduplicate_beliefs(belief_set)
        action = memo[(belief_set, remaining)][1]
        if not action.startswith("query_"):
            return action, None
        source_index = int(action.split("_")[1])
        next_remaining = tuple(
            j for j in remaining if j != source_index
        )
        children = tuple(
            recurse(
                rectangular_child_belief_set(
                    belief_set,
                    source_channel_sets[source_index],
                    evidence,
                ),
                next_remaining,
            )
            for evidence in range(alphabet_size)
        )
        return action, children

    return recurse((float(prior_y1),), tuple(range(n_sources)))

In [ ]:

# Static finite-scenario double oracle



def static_best_response_general(
    prior_y1: float,
    scenario_channels: np.ndarray,
    costs: np.ndarray,
    abstention_cost: float,
    adversary_distribution: np.ndarray,
    max_states: int | None = MAX_MEMO_STATES,
):
    scenario_channels = np.asarray(scenario_channels, dtype=float)
    costs = np.asarray(costs, dtype=float)
    adversary_distribution = np.asarray(adversary_distribution, dtype=float)
    n_models, n_sources, n_hypotheses, alphabet_size = scenario_channels.shape
    if n_hypotheses != 2:
        raise ValueError("This implementation assumes binary hypotheses.")
    if len(adversary_distribution) != n_models:
        raise ValueError("adversary distribution length does not match K.")

    memo = {}

    def likelihood_matrix(history: tuple[tuple[int, int], ...]) -> np.ndarray:
        likelihood = np.tile(
            np.array([1.0 - prior_y1, prior_y1], dtype=float),
            (n_models, 1),
        )
        for source_index, evidence in history:
            likelihood *= scenario_channels[
                :, source_index, :, evidence
            ]
        return likelihood

    def recurse(history, remaining):
        key = (history, remaining)
        if key in memo:
            return memo[key]
        _memo_state_guard(memo, max_states)

        likelihood = likelihood_matrix(history)
        reach_probability = likelihood.sum(axis=1)
        candidates = [
            ("answer_0", likelihood[:, 1].copy(), None),
            ("answer_1", likelihood[:, 0].copy(), None),
            (
                "abstain",
                abstention_cost * reach_probability.copy(),
                None,
            ),
        ]

        for source_index in remaining:
            next_remaining = tuple(
                j for j in remaining if j != source_index
            )
            child_results = [
                recurse(
                    history + ((int(source_index), int(evidence)),),
                    next_remaining,
                )
                for evidence in range(alphabet_size)
            ]
            risk_vector = costs[source_index] * reach_probability
            for child in child_results:
                risk_vector = risk_vector + child[1]
            candidates.append(
                (
                    f"query_{source_index}",
                    risk_vector,
                    tuple(child[2] for child in child_results),
                )
            )

        weighted_values = [
            float(np.dot(adversary_distribution, candidate[1]))
            for candidate in candidates
        ]
        selected_index = int(np.argmin(weighted_values))
        action, risk_vector, children = candidates[selected_index]
        memo[key] = (
            float(weighted_values[selected_index]),
            np.asarray(risk_vector, dtype=float),
            (action, children),
        )
        return memo[key]

    root = recurse(tuple(), tuple(range(n_sources)))
    return root, memo


def solve_restricted_adversary_general(risk_matrix: np.ndarray):
    risk_matrix = np.asarray(risk_matrix, dtype=float)
    n_policies, n_models = risk_matrix.shape

    objective = np.zeros(n_models + 1, dtype=float)
    objective[-1] = -1.0

    inequality_matrix = np.zeros(
        (n_policies, n_models + 1),
        dtype=float,
    )
    inequality_matrix[:, :n_models] = -risk_matrix
    inequality_matrix[:, -1] = 1.0
    inequality_rhs = np.zeros(n_policies, dtype=float)

    equality_matrix = np.zeros((1, n_models + 1), dtype=float)
    equality_matrix[0, :n_models] = 1.0
    equality_rhs = np.array([1.0], dtype=float)

    bounds = [(0.0, None)] * n_models + [(None, None)]
    result = linprog(
        objective,
        A_ub=inequality_matrix,
        b_ub=inequality_rhs,
        A_eq=equality_matrix,
        b_eq=equality_rhs,
        bounds=bounds,
        method="highs",
    )
    if not result.success:
        raise RuntimeError(f"Restricted adversary LP failed: {result.message}")
    return float(result.x[-1]), np.asarray(result.x[:n_models], dtype=float)


def solve_restricted_policy_mixture_general(
    policy_risk_matrix: np.ndarray,
):
    policy_risk_matrix = np.asarray(policy_risk_matrix, dtype=float)
    n_policies, n_models = policy_risk_matrix.shape

    objective = np.zeros(n_policies + 1, dtype=float)
    objective[-1] = 1.0

    inequality_matrix = np.zeros(
        (n_models, n_policies + 1),
        dtype=float,
    )
    inequality_matrix[:, :n_policies] = policy_risk_matrix.T
    inequality_matrix[:, -1] = -1.0
    inequality_rhs = np.zeros(n_models, dtype=float)

    equality_matrix = np.zeros((1, n_policies + 1), dtype=float)
    equality_matrix[0, :n_policies] = 1.0
    equality_rhs = np.array([1.0], dtype=float)

    bounds = [(0.0, None)] * n_policies + [(None, None)]
    result = linprog(
        objective,
        A_ub=inequality_matrix,
        b_ub=inequality_rhs,
        A_eq=equality_matrix,
        b_eq=equality_rhs,
        bounds=bounds,
        method="highs",
    )
    if not result.success:
        raise RuntimeError(f"Restricted policy-mixture LP failed: {result.message}")

    weights = np.asarray(result.x[:n_policies], dtype=float).copy()
    if not np.all(np.isfinite(weights)) or np.min(weights) < -1e-8:
        raise RuntimeError("Policy-mixture LP returned invalid policy weights.")
    # Repair only negative solver roundoff; retain every positive policy mass.
    weights = np.maximum(weights, 0.0)
    mass = float(weights.sum())
    if mass <= 0.0 or abs(mass - 1.0) > 1e-8:
        raise RuntimeError("Policy-mixture weights do not sum to one within tolerance.")
    weights /= mass
    value = float(np.max(policy_risk_matrix.T @ weights))
    if abs(value - float(result.x[-1])) > 1e-8:
        raise RuntimeError("Implemented policy mixture disagrees with the LP objective.")
    return weights, value


def static_double_oracle_general(
    prior_y1: float,
    scenario_channels: np.ndarray,
    costs: np.ndarray,
    abstention_cost: float,
    tolerance: float = STATIC_TOLERANCE,
    max_iterations: int = MAX_DO_ITERATIONS,
    max_states: int | None = MAX_MEMO_STATES,
):
    scenario_channels = np.asarray(scenario_channels, dtype=float)
    n_models = scenario_channels.shape[0]
    adversary_distribution = np.ones(n_models, dtype=float) / n_models

    first_response, first_memo = static_best_response_general(
        prior_y1,
        scenario_channels,
        costs,
        abstention_cost,
        adversary_distribution,
        max_states=max_states,
    )
    first_state_count = len(first_memo)
    first_max_depth = max(
        (len(key[0]) for key in first_memo),
        default=0,
    )
    del first_memo

    policy_risk_vectors = [first_response[1]]
    policy_trees = [first_response[2]]
    observed_vectors = {tuple(np.round(first_response[1], 12))}
    convergence_history = []
    best_response_state_counts = [int(first_state_count)]
    best_response_max_depths = [int(first_max_depth)]
    stalled = False

    for iteration in range(max_iterations):
        risk_matrix = np.vstack(policy_risk_vectors)
        upper_bound, adversary_distribution = solve_restricted_adversary_general(
            risk_matrix
        )
        best_response, best_memo = static_best_response_general(
            prior_y1,
            scenario_channels,
            costs,
            abstention_cost,
            adversary_distribution,
            max_states=max_states,
        )
        best_response_state_counts.append(int(len(best_memo)))
        best_response_max_depths.append(
            int(max((len(key[0]) for key in best_memo), default=0))
        )
        del best_memo

        lower_bound = float(
            np.dot(adversary_distribution, best_response[1])
        )
        gap = float(upper_bound - lower_bound)
        convergence_history.append(
            {
                "iteration": int(iteration),
                "upper_bound": float(upper_bound),
                "lower_bound": float(lower_bound),
                "gap": float(gap),
                "n_policy_columns": int(len(policy_risk_vectors)),
            }
        )

        if gap <= tolerance:
            break

        vector_key = tuple(np.round(best_response[1], 12))
        if vector_key in observed_vectors:
            stalled = True
            break
        observed_vectors.add(vector_key)
        policy_risk_vectors.append(best_response[1])
        policy_trees.append(best_response[2])
    else:
        raise RuntimeError("Static double oracle reached iteration limit.")

    risk_matrix = np.vstack(policy_risk_vectors)
    policy_weights, restricted_value = solve_restricted_policy_mixture_general(
        risk_matrix
    )
    final_value = float(convergence_history[-1]["upper_bound"])
    if not np.isclose(final_value, restricted_value, atol=1e-7):
        raise AssertionError(
            "Static primal and dual values disagree: "
            f"{final_value} vs {restricted_value}"
        )

    return {
        "value": final_value,
        "policy_weights": policy_weights,
        "policy_trees": policy_trees,
        "adversary_distribution": adversary_distribution,
        "policy_risk_matrix": risk_matrix,
        "convergence_history": convergence_history,
        "best_response_state_counts": best_response_state_counts,
        "best_response_max_depths": best_response_max_depths,
        "stalled": bool(stalled),
    }

## Policy reconstruction and exact outcome evaluation

The solver benchmark and the policy-evaluation benchmark are reported separately. This prevents the (A^m) outcome enumeration used for exact tree evaluation from being confused with the DP state-space cost.

In [ ]:

# Policy trees, outcome tensors, and exact evaluation



def all_outcomes(n_sources: int, alphabet_size: int) -> np.ndarray:
    return np.asarray(
        list(itertools.product(range(alphabet_size), repeat=n_sources)),
        dtype=int,
    )


def tree_tensor_general(
    tree,
    costs: np.ndarray,
    abstention_cost: float,
    outcomes: np.ndarray,
) -> np.ndarray:
    costs = np.asarray(costs, dtype=float)
    n_outcomes = len(outcomes)
    tensor = np.zeros(
        (2, n_outcomes, len(METRIC_NAMES)),
        dtype=float,
    )

    for outcome_index, evidence_vector in enumerate(outcomes):
        node = tree
        total_cost = 0.0
        n_queries = 0.0
        steps = 0
        while isinstance(node[0], str) and node[0].startswith("query_"):
            source_index = int(node[0].split("_")[1])
            total_cost += float(costs[source_index])
            n_queries += 1.0
            evidence = int(evidence_vector[source_index])
            node = node[1][evidence]
            steps += 1
            if steps > len(costs):
                raise RuntimeError("Policy tree exceeded the no-replacement horizon.")

        action = node[0]
        is_abstain = float(action == "abstain")
        for y in (0, 1):
            is_wrong = float(
                (not is_abstain) and action != f"answer_{y}"
            )
            tensor[y, outcome_index] = [
                is_wrong + abstention_cost * is_abstain + total_cost,
                is_wrong,
                is_abstain,
                1.0 - is_abstain,
                total_cost,
                n_queries,
            ]
    return tensor


def scenario_distribution_general(
    prior_y1: float,
    channels: np.ndarray,
    outcomes: np.ndarray,
) -> np.ndarray:
    channels = np.asarray(channels, dtype=float)
    if channels.ndim == 3:
        channels = channels[None, ...]
    n_models, n_sources, _, alphabet_size = channels.shape
    if outcomes.shape[1] != n_sources:
        raise ValueError("outcome dimension does not match source count.")

    probabilities = np.ones(
        (n_models, 2, len(outcomes)),
        dtype=float,
    )
    for source_index in range(n_sources):
        factors = np.stack(
            [
                channels[:, source_index, :, int(evidence)]
                for evidence in outcomes[:, source_index]
            ],
            axis=-1,
        )
        probabilities *= factors

    probabilities *= np.array(
        [1.0 - prior_y1, prior_y1],
        dtype=float,
    )[None, :, None]
    if not np.allclose(
        probabilities.sum(axis=(1, 2)),
        1.0,
        atol=1e-10,
    ):
        raise AssertionError("Joint outcome probabilities are not normalized.")
    return probabilities


def evaluate_tensor_general(
    tensor: np.ndarray,
    prior_y1: float,
    channels: np.ndarray,
    outcomes: np.ndarray,
) -> np.ndarray:
    probabilities = scenario_distribution_general(
        prior_y1,
        channels,
        outcomes,
    )
    return np.einsum("kyo,yom->km", probabilities, tensor)


def _max_tree_depth(tree) -> int:
    if tree is None or not tree[0].startswith("query_"):
        return 0
    return 1 + max(_max_tree_depth(child) for child in tree[1])


def _tree_query_count(tree) -> int:
    if tree is None or not tree[0].startswith("query_"):
        return 0
    return 1 + max(_tree_query_count(child) for child in tree[1])


def reconstruct_and_evaluate(
    method: str,
    result,
    problem: dict,
):
    prior_y1 = problem["prior_y1"]
    abstention_cost = problem["abstention_cost"]
    costs = problem["costs"]
    robust_channels = problem["channels"]
    nominal_channels = problem["nominal_channels"]
    n_sources = problem["n_sources"]
    alphabet_size = problem["alphabet_size"]
    outcomes = all_outcomes(n_sources, alphabet_size)

    reconstruction_start = time.perf_counter()
    if method == "NominalDP":
        tree = nominal_tree_from_memo(
            prior_y1,
            nominal_channels,
            result["memo"],
        )
        trees = [tree]
    elif method == "HCR":
        tree = hcr_tree_from_memo(
            result["memo"],
            n_sources,
            alphabet_size,
        )
        trees = [tree]
    elif method == "Rectangular":
        tree = rectangular_tree_from_memo(
            prior_y1,
            result["source_channel_sets"],
            result["memo"],
            n_sources,
            alphabet_size,
        )
        trees = [tree]
    elif method == "StaticDO":
        trees = result["policy_trees"]
    else:
        raise ValueError(f"Unknown method: {method}")
    reconstruction_seconds = time.perf_counter() - reconstruction_start

    evaluation_start = time.perf_counter()
    tensors = [
        tree_tensor_general(
            tree,
            costs,
            abstention_cost,
            outcomes,
        )
        for tree in trees
    ]
    if method == "StaticDO":
        tensor = sum(
            weight * tree_tensor
            for weight, tree_tensor in zip(
                result["policy_weights"],
                tensors,
            )
        )
    else:
        tensor = tensors[0]

    evaluation_channels = (
        nominal_channels if method == "NominalDP" else robust_channels
    )
    values = evaluate_tensor_general(
        tensor,
        prior_y1,
        evaluation_channels,
        outcomes,
    )
    evaluation_seconds = time.perf_counter() - evaluation_start

    return {
        "tree": trees[0] if trees else None,
        "n_policy_trees": int(len(trees)),
        "outcome_count": int(len(outcomes)),
        "tree_depth": int(
            max((_max_tree_depth(tree) for tree in trees), default=0)
        ),
        "max_queries": int(
            max((_tree_query_count(tree) for tree in trees), default=0)
        ),
        "reconstruction_seconds": float(reconstruction_seconds),
        "evaluation_seconds": float(evaluation_seconds),
        "values": values,
        "mean_risk": float(np.mean(values[:, 0])),
        "worst_model_risk": float(np.max(values[:, 0])),
        "mean_queries": float(np.mean(values[:, 5])),
        "mean_cost": float(np.mean(values[:, 4])),
    }

## Runtime and memory instrumentation

RSS is sampled in a lightweight background thread. When available, `/proc/self/status` or `psutil` provides resident-memory observations. The benchmark also records Python allocation peaks when `MEASURE_PYTHON_ALLOCATIONS=True`. A timeout or state-budget event is preserved as a first-class result rather than converted to a missing value.

In [ ]:

# Runtime, memory, timeout, and state diagnostics



def _read_rss_bytes():
    if psutil is not None:
        try:
            return int(psutil.Process(os.getpid()).memory_info().rss)
        except Exception:
            pass

    status_path = Path("/proc/self/status")
    if status_path.exists():
        try:
            for line in status_path.read_text(encoding="utf-8").splitlines():
                if line.startswith("VmRSS:"):
                    parts = line.split()
                    return int(parts[1]) * 1024
        except Exception:
            pass
    return None


class PeakRSSSampler:
    def __init__(self, interval_seconds: float = RSS_POLL_SECONDS):
        self.interval_seconds = float(interval_seconds)
        self.start_bytes = None
        self.peak_bytes = None
        self._stop = threading.Event()
        self._thread = None

    def _sample(self):
        value = _read_rss_bytes()
        if value is not None:
            self.peak_bytes = max(self.peak_bytes or value, value)

    def __enter__(self):
        self.start_bytes = _read_rss_bytes()
        self.peak_bytes = self.start_bytes
        if self.start_bytes is not None:
            self._thread = threading.Thread(
                target=self._run,
                name="rose-rag-rss-sampler",
                daemon=True,
            )
            self._thread.start()
        return self

    def _run(self):
        while not self._stop.is_set():
            self._sample()
            self._stop.wait(self.interval_seconds)

    def __exit__(self, exc_type, exc_value, exc_traceback):
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=1.0)
        self._sample()

    @property
    def peak_mb(self):
        if self.peak_bytes is None:
            return np.nan
        return float(self.peak_bytes / (1024.0 ** 2))

    @property
    def delta_mb(self):
        if self.start_bytes is None or self.peak_bytes is None:
            return np.nan
        return float(max(0, self.peak_bytes - self.start_bytes) / (1024.0 ** 2))


@contextmanager
def deadline(seconds: float | None):
    if seconds is None or seconds <= 0:
        yield
        return
    if threading.current_thread() is not threading.main_thread():
        yield
        return

    previous_handler = signal.getsignal(signal.SIGALRM)

    def handler(signum, frame):
        raise BenchmarkTimeout(f"case exceeded {seconds} seconds")

    signal.signal(signal.SIGALRM, handler)
    signal.setitimer(signal.ITIMER_REAL, float(seconds))
    try:
        yield
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0.0)
        signal.signal(signal.SIGALRM, previous_handler)


def _trace_allocation_start():
    if MEASURE_PYTHON_ALLOCATIONS:
        tracemalloc.start()


def _trace_allocation_stop():
    if not MEASURE_PYTHON_ALLOCATIONS:
        return np.nan
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return float(peak / (1024.0 ** 2))


def measure_one_call(fn, timeout_seconds: float | None):
    sampler = PeakRSSSampler()
    result = None
    error = None
    status = "completed"
    start = time.perf_counter()
    _trace_allocation_start()
    with sampler:
        try:
            with deadline(timeout_seconds):
                result = fn()
        except StateBudgetExceeded as exc:
            status = "state_budget_exceeded"
            error = str(exc)
        except BenchmarkTimeout as exc:
            status = "timeout"
            error = str(exc)
        except MemoryError as exc:
            status = "memory_error"
            error = repr(exc)
        except Exception as exc:
            status = "error"
            error = "".join(
                traceback.format_exception_only(type(exc), exc)
            ).strip()
    elapsed = time.perf_counter() - start
    allocation_peak = _trace_allocation_stop()

    if np.isfinite(sampler.peak_mb) and sampler.peak_mb > MAX_PEAK_MEMORY_MB:
        status = "memory_limit_exceeded"
        error = (
            f"peak RSS {sampler.peak_mb:.2f} MB exceeded "
            f"limit {MAX_PEAK_MEMORY_MB:.2f} MB"
        )

    return result, {
        "status": status,
        "error": error,
        "wall_time_seconds": float(elapsed),
        "peak_rss_mb": sampler.peak_mb,
        "peak_rss_delta_mb": sampler.delta_mb,
        "python_alloc_peak_mb": allocation_peak,
    }


def _solver_summary(method: str, result) -> dict:
    if method == "NominalDP":
        root = result["root"]
        memo = result["memo"]
        return {
            "solver_value": float(root[0]),
            "memo_states": int(len(memo)),
            "action_evaluations": int(
                sum(len(value[2]) for value in memo.values())
            ),
            "max_history_depth": int(
                max(
                    (len(key[1]) for key in memo),
                    default=0,
                )
            ),
            "rect_total_beliefs": np.nan,
            "rect_max_belief_set": np.nan,
            "static_do_iterations": np.nan,
            "static_policy_columns": np.nan,
            "static_gap": np.nan,
            "static_lp_adversary_rows": np.nan,
            "static_lp_adversary_variables": np.nan,
            "static_lp_primal_rows": np.nan,
            "static_lp_primal_variables": np.nan,
            "static_br_states_total": np.nan,
            "static_br_states_max": np.nan,
        }

    if method == "HCR":
        root = result["root"]
        memo = result["memo"]
        return {
            "solver_value": float(root[0]),
            "memo_states": int(len(memo)),
            "action_evaluations": int(
                sum(len(value[2]) for value in memo.values())
            ),
            "max_history_depth": int(
                max((len(key[0]) for key in memo), default=0)
            ),
            "rect_total_beliefs": np.nan,
            "rect_max_belief_set": np.nan,
            "static_do_iterations": np.nan,
            "static_policy_columns": np.nan,
            "static_gap": np.nan,
            "static_lp_adversary_rows": np.nan,
            "static_lp_adversary_variables": np.nan,
            "static_lp_primal_rows": np.nan,
            "static_lp_primal_variables": np.nan,
            "static_br_states_total": np.nan,
            "static_br_states_max": np.nan,
        }

    if method == "Rectangular":
        root = result["root"]
        memo = result["memo"]
        source_channel_sets = result["source_channel_sets"]
        belief_lengths = [len(key[0]) for key in memo]
        return {
            "solver_value": float(root[0]),
            "memo_states": int(len(memo)),
            "action_evaluations": int(
                sum(len(value[2]) for value in memo.values())
            ),
            "max_history_depth": int(
                max(
                    (len(key[1]) for key in memo),
                    default=0,
                )
            ),
            "rect_total_beliefs": int(sum(belief_lengths)),
            "rect_max_belief_set": int(max(belief_lengths, default=0)),
            "rect_projection_cardinality_max": int(
                max((len(items) for items in source_channel_sets), default=0)
            ),
            "static_do_iterations": np.nan,
            "static_policy_columns": np.nan,
            "static_gap": np.nan,
            "static_lp_adversary_rows": np.nan,
            "static_lp_adversary_variables": np.nan,
            "static_lp_primal_rows": np.nan,
            "static_lp_primal_variables": np.nan,
            "static_br_states_total": np.nan,
            "static_br_states_max": np.nan,
        }

    if method == "StaticDO":
        history = result["convergence_history"]
        risk_matrix = result["policy_risk_matrix"]
        br_counts = result["best_response_state_counts"]
        br_depths = result["best_response_max_depths"]
        return {
            "solver_value": float(result["value"]),
            "memo_states": int(max(br_counts, default=0)),
            "action_evaluations": np.nan,
            "max_history_depth": int(max(br_depths, default=0)),
            "rect_total_beliefs": np.nan,
            "rect_max_belief_set": np.nan,
            "static_do_iterations": int(len(history)),
            "static_policy_columns": int(risk_matrix.shape[0]),
            "static_gap": float(history[-1]["gap"] if history else np.nan),
            "static_lp_adversary_rows": int(risk_matrix.shape[0]),
            "static_lp_adversary_variables": int(risk_matrix.shape[1] + 1),
            "static_lp_primal_rows": int(risk_matrix.shape[1]),
            "static_lp_primal_variables": int(risk_matrix.shape[0] + 1),
            "static_br_states_total": int(sum(br_counts)),
            "static_br_states_max": int(max(br_counts, default=0)),
        }

    raise ValueError(f"Unknown method: {method}")


def _empty_method_row(config: dict, method: str) -> dict:
    return {
        "sweep": config["sweep"],
        "n_sources": config["n_sources"],
        "alphabet_size": config["alphabet_size"],
        "n_models": config["n_models"],
        "seed": config["seed"],
        "method": method,
        "status": "not_run",
        "solver_status": "not_run",
        "evaluation_status": "not_run",
        "error": None,
        "solver_error": None,
        "evaluation_error": None,
        "measurement_scope": (
            "isolated_child_process"
            if ISOLATE_METHOD_RUNS
            else "long_lived_jupyter_kernel"
        ),
        "repeat_count": 0,
        "wall_time_seconds": np.nan,
        "wall_time_min_seconds": np.nan,
        "wall_time_max_seconds": np.nan,
        "total_wall_time_seconds": np.nan,
        "total_wall_time_min_seconds": np.nan,
        "total_wall_time_max_seconds": np.nan,
        "peak_rss_mb": np.nan,
        "peak_rss_delta_mb": np.nan,
        "solver_peak_rss_mb": np.nan,
        "solver_peak_rss_delta_mb": np.nan,
        "python_alloc_peak_mb": np.nan,
        "solver_value": np.nan,
        "memo_states": np.nan,
        "action_evaluations": np.nan,
        "max_history_depth": np.nan,
        "rect_total_beliefs": np.nan,
        "rect_max_belief_set": np.nan,
        "rect_projection_cardinality_max": np.nan,
        "static_do_iterations": np.nan,
        "static_policy_columns": np.nan,
        "static_gap": np.nan,
        "static_lp_adversary_rows": np.nan,
        "static_lp_adversary_variables": np.nan,
        "static_lp_primal_rows": np.nan,
        "static_lp_primal_variables": np.nan,
        "static_br_states_total": np.nan,
        "static_br_states_max": np.nan,
        "n_policy_trees": np.nan,
        "outcome_count": np.nan,
        "tree_depth": np.nan,
        "max_queries": np.nan,
        "reconstruction_seconds": np.nan,
        "evaluation_seconds": np.nan,
        "mean_risk": np.nan,
        "worst_model_risk": np.nan,
        "mean_queries": np.nan,
        "mean_cost": np.nan,
        "hierarchy_ok": np.nan,
        "static_gap_ok": np.nan,
        "static_value_consistency_error": np.nan,
    }

In [ ]:

# Case execution and result assembly



def _case_config(sweep: str, n_sources: int, alphabet_size: int, n_models: int):
    return {
        "sweep": sweep,
        "n_sources": int(n_sources),
        "alphabet_size": int(alphabet_size),
        "n_models": int(n_models),
        "seed": int(BASE_SEED),
    }


def build_case_grid() -> list[dict]:
    configs = []
    source_values = SOURCE_VALUES if RUN_HIGH_SOURCE_CASES else tuple(
        m for m in SOURCE_VALUES if m <= 6
    )
    configs.extend(
        _case_config("source_scaling", m, 2, 3)
        for m in source_values
    )
    if RUN_ALPHABET_SWEEP:
        configs.extend(
            _case_config("alphabet_scaling", 3, alphabet_size, 3)
            for alphabet_size in ALPHABET_VALUES
        )
    if RUN_MODEL_SWEEP:
        configs.extend(
            _case_config("model_scaling", 3, 2, k)
            for k in MODEL_VALUES
        )

    unique = {}
    for config in configs:
        key = (
            config["sweep"],
            config["n_sources"],
            config["alphabet_size"],
            config["n_models"],
        )
        unique[key] = config
    return list(unique.values())


def _repeat_count_for_config(config: dict) -> int:
    if config["n_sources"] <= SMALL_CASE_REPEAT_LIMIT_M:
        return int(SMALL_CASE_REPEATS)
    return 1


def _solver_callable(method: str, problem: dict):
    prior_y1 = problem["prior_y1"]
    abstention_cost = problem["abstention_cost"]
    costs = problem["costs"]
    channels = problem["channels"]

    if method == "NominalDP":
        return lambda: nominal_dp_general(
            prior_y1,
            problem["nominal_channels"],
            costs,
            abstention_cost,
            max_states=MAX_MEMO_STATES,
        )

    if method == "HCR":
        return lambda: hcr_dp_general(
            prior_y1,
            channels,
            costs,
            abstention_cost,
            max_states=MAX_MEMO_STATES,
        )

    if method == "Rectangular":
        return lambda: rectangular_dp_general(
            prior_y1,
            channels,
            costs,
            abstention_cost,
            max_states=MAX_MEMO_STATES,
        )

    if method == "StaticDO":
        uniform = np.ones(problem["n_models"], dtype=float)
        uniform /= uniform.sum()
        return lambda: static_double_oracle_general(
            prior_y1,
            channels,
            costs,
            abstention_cost,
            tolerance=STATIC_TOLERANCE,
            max_iterations=MAX_DO_ITERATIONS,
            max_states=MAX_MEMO_STATES,
        )

    raise ValueError(f"Unknown method: {method}")


def _normalize_solver_result(method: str, raw_result):
    if method == "NominalDP":
        root, memo = raw_result
        return {
            "root": root,
            "memo": memo,
        }
    if method == "HCR":
        root, memo = raw_result
        return {
            "root": root,
            "memo": memo,
        }
    if method == "Rectangular":
        root, memo, source_channel_sets = raw_result
        return {
            "root": root,
            "memo": memo,
            "source_channel_sets": source_channel_sets,
        }
    if method == "StaticDO":
        return raw_result
    raise ValueError(method)


def _evaluation_metrics(evaluation: dict) -> dict:
    """Keep only scalar evaluation metrics for inter-process transfer."""
    return {
        key: value
        for key, value in evaluation.items()
        if key not in {"tree", "values"}
    }


def _worker_failure_payload(status: str, error: str) -> dict:
    return {
        "solver_status": status,
        "evaluation_status": "not_run",
        "status": status,
        "solver_error": error,
        "evaluation_error": None,
        "error": error,
        "summary": None,
        "evaluation": None,
        "measurement": {
            "status": status,
            "solver_status": status,
            "evaluation_status": "not_run",
            "error": error,
            "wall_time_seconds": np.nan,
            "total_wall_time_seconds": np.nan,
            "peak_rss_mb": np.nan,
            "peak_rss_delta_mb": np.nan,
            "solver_peak_rss_mb": np.nan,
            "solver_peak_rss_delta_mb": np.nan,
            "python_alloc_peak_mb": np.nan,
        },
    }


def _isolated_case_worker(conn, method: str, problem: dict):
    """Run one complete method/configuration in a disposable child process.

    Only scalar summaries cross the pipe. The potentially large DP memo and
    policy trees die with the child, so they cannot inflate the next case's
    baseline RSS in the notebook kernel.
    """
    payload = {
        "solver_status": "not_run",
        "evaluation_status": "not_run",
        "status": "not_run",
        "solver_error": None,
        "evaluation_error": None,
        "error": None,
        "summary": None,
        "evaluation": None,
    }
    raw_result = None
    solver_finished = False
    solver_wall_time = np.nan
    total_start = time.perf_counter()
    total_sampler = PeakRSSSampler()
    solver_sampler = PeakRSSSampler()
    _trace_allocation_start()

    try:
        with total_sampler:
            try:
                # One deadline covers the full solver + reconstruction + exact
                # evaluation path. A hard parent-side join below protects
                # against a native-code call that ignores SIGALRM.
                with deadline(MAX_CASE_SECONDS):
                    solver_start = time.perf_counter()
                    with solver_sampler:
                        raw_result = _solver_callable(method, problem)()
                    solver_wall_time = time.perf_counter() - solver_start
                    solver_finished = True
                    payload["solver_status"] = "completed"

                    representative = _normalize_solver_result(
                        method,
                        raw_result,
                    )
                    payload["summary"] = _solver_summary(
                        method,
                        representative,
                    )

                    evaluation_start = time.perf_counter()
                    evaluation = reconstruct_and_evaluate(
                        method,
                        representative,
                        problem,
                    )
                    evaluation["evaluation_seconds"] = float(
                        time.perf_counter() - evaluation_start
                    )
                    payload["evaluation"] = _evaluation_metrics(evaluation)
                    payload["evaluation_status"] = "completed"

            except StateBudgetExceeded as exc:
                if solver_finished:
                    payload["evaluation_status"] = "state_budget_exceeded"
                    payload["evaluation_error"] = str(exc)
                else:
                    payload["solver_status"] = "state_budget_exceeded"
                    payload["solver_error"] = str(exc)
            except BenchmarkTimeout as exc:
                if solver_finished:
                    payload["evaluation_status"] = "timeout"
                    payload["evaluation_error"] = str(exc)
                else:
                    payload["solver_status"] = "timeout"
                    payload["solver_error"] = str(exc)
            except MemoryError as exc:
                if solver_finished:
                    payload["evaluation_status"] = "memory_error"
                    payload["evaluation_error"] = repr(exc)
                else:
                    payload["solver_status"] = "memory_error"
                    payload["solver_error"] = repr(exc)
            except Exception as exc:
                error_text = "".join(
                    traceback.format_exception_only(type(exc), exc)
                ).strip()
                if solver_finished:
                    payload["evaluation_status"] = "error"
                    payload["evaluation_error"] = error_text
                else:
                    payload["solver_status"] = "error"
                    payload["solver_error"] = error_text

    finally:
        allocation_peak = _trace_allocation_stop()
        if payload["solver_status"] != "completed":
            payload["status"] = payload["solver_status"]
        elif payload["evaluation_status"] != "completed":
            payload["status"] = "evaluation_" + str(
                payload["evaluation_status"]
            )
        else:
            payload["status"] = "completed"

        payload["error"] = (
            payload["solver_error"] or payload["evaluation_error"]
        )
        payload["measurement"] = {
            "status": payload["status"],
            "solver_status": payload["solver_status"],
            "evaluation_status": payload["evaluation_status"],
            "error": payload["error"],
            "wall_time_seconds": float(solver_wall_time),
            "total_wall_time_seconds": float(
                time.perf_counter() - total_start
            ),
            "peak_rss_mb": total_sampler.peak_mb,
            "peak_rss_delta_mb": total_sampler.delta_mb,
            "solver_peak_rss_mb": solver_sampler.peak_mb,
            "solver_peak_rss_delta_mb": solver_sampler.delta_mb,
            "python_alloc_peak_mb": allocation_peak,
        }
        try:
            conn.send(payload)
        except Exception:
            # The parent will report a worker/transport failure if the child
            # cannot transfer its scalar diagnostic payload.
            pass
        finally:
            conn.close()


def _inprocess_case_call(method: str, problem: dict) -> dict:
    """Debug fallback; its RSS values are explicitly kernel-scoped."""
    raw_result, measurement = measure_one_call(
        _solver_callable(method, problem),
        timeout_seconds=MAX_CASE_SECONDS,
    )
    payload = {
        "solver_status": measurement["status"],
        "evaluation_status": "not_run",
        "status": measurement["status"],
        "solver_error": measurement["error"],
        "evaluation_error": None,
        "error": measurement["error"],
        "summary": None,
        "evaluation": None,
    }
    total_start = time.perf_counter()
    if measurement["status"] == "completed":
        payload["solver_status"] = "completed"
        try:
            representative = _normalize_solver_result(method, raw_result)
            payload["summary"] = _solver_summary(method, representative)
            evaluation = reconstruct_and_evaluate(
                method,
                representative,
                problem,
            )
            payload["evaluation"] = _evaluation_metrics(evaluation)
            payload["evaluation_status"] = "completed"
            payload["status"] = "completed"
            payload["error"] = None
        except Exception as exc:
            payload["evaluation_status"] = "error"
            payload["status"] = "evaluation_error"
            payload["evaluation_error"] = "".join(
                traceback.format_exception_only(type(exc), exc)
            ).strip()
            payload["error"] = payload["evaluation_error"]
    measurement = dict(measurement)
    measurement["solver_status"] = payload["solver_status"]
    measurement["evaluation_status"] = payload["evaluation_status"]
    measurement["status"] = payload["status"]
    measurement["total_wall_time_seconds"] = float(
        time.perf_counter() - total_start + measurement["wall_time_seconds"]
    )
    measurement["solver_peak_rss_mb"] = measurement["peak_rss_mb"]
    measurement["solver_peak_rss_delta_mb"] = measurement[
        "peak_rss_delta_mb"
    ]
    payload["measurement"] = measurement
    return payload


def _isolated_case_call(method: str, problem: dict) -> dict:
    if not ISOLATE_METHOD_RUNS:
        return _inprocess_case_call(method, problem)

    available = mp.get_all_start_methods()
    if MP_START_METHOD not in available:
        return _worker_failure_payload(
            "isolation_error",
            f"Requested multiprocessing start method {MP_START_METHOD!r} "
            f"is unavailable; available methods: {available}",
        )

    context = mp.get_context(MP_START_METHOD)
    parent_conn, child_conn = context.Pipe(duplex=False)
    process = context.Process(
        target=_isolated_case_worker,
        args=(child_conn, method, problem),
        name=f"rose-rag-{method}",
    )
    process.daemon = True
    try:
        process.start()
    except Exception as exc:
        child_conn.close()
        parent_conn.close()
        return _worker_failure_payload(
            "isolation_error",
            "Could not start isolated worker: "
            + "".join(traceback.format_exception_only(type(exc), exc)).strip(),
        )
    finally:
        child_conn.close()

    # The worker has the same hard deadline as the computation. The grace
    # interval covers pipe transfer and orderly interpreter shutdown.
    process.join(float(MAX_CASE_SECONDS) + PROCESS_GRACE_SECONDS)
    payload = None
    if parent_conn.poll(1.0):
        try:
            payload = parent_conn.recv()
        except Exception:
            payload = None
    parent_conn.close()

    if process.is_alive():
        process.terminate()
        process.join(5.0)
        return _worker_failure_payload(
            "timeout",
            "Isolated worker exceeded the hard process deadline and was "
            "terminated by the parent.",
        )
    if payload is None:
        return _worker_failure_payload(
            "worker_error",
            f"Isolated worker exited without a diagnostic payload "
            f"(exitcode={process.exitcode}).",
        )
    return payload


def run_one_case(config: dict):
    problem = make_benchmark_problem(
        n_sources=config["n_sources"],
        alphabet_size=config["alphabet_size"],
        n_models=config["n_models"],
        seed=config["seed"],
    )

    rows = []
    repeat_count = _repeat_count_for_config(config)

    for method in METHODS:
        row = _empty_method_row(config, method)
        row["repeat_count"] = int(repeat_count)
        all_records = []
        successful_payloads = []
        solver_status = "completed"
        evaluation_status = "not_run"
        solver_error = None
        evaluation_error = None

        for repetition in range(repeat_count):
            payload = _isolated_case_call(method, problem)
            measurement = payload["measurement"]
            all_records.append(measurement)
            if payload["solver_status"] != "completed":
                solver_status = payload["solver_status"]
                solver_error = payload["solver_error"]
                break
            successful_payloads.append(payload)
            evaluation_status = payload["evaluation_status"]
            evaluation_error = payload["evaluation_error"]

        row["solver_status"] = solver_status
        row["evaluation_status"] = evaluation_status
        row["solver_error"] = solver_error
        row["evaluation_error"] = evaluation_error
        row["status"] = (
            solver_status
            if solver_status != "completed"
            else (
                "completed"
                if evaluation_status == "completed"
                else "evaluation_" + str(evaluation_status)
            )
        )
        row["error"] = solver_error or evaluation_error
        if all_records:
            successful = [
                record
                for record in all_records
                if record["solver_status"] == "completed"
            ]
            if successful:
                times = [record["wall_time_seconds"] for record in successful]
                total_times = [
                    record["total_wall_time_seconds"]
                    for record in successful
                ]
                rss_peaks = [record["peak_rss_mb"] for record in successful]
                rss_deltas = [
                    record["peak_rss_delta_mb"] for record in successful
                ]
                solver_rss_peaks = [
                    record["solver_peak_rss_mb"]
                    for record in successful
                ]
                solver_rss_deltas = [
                    record["solver_peak_rss_delta_mb"]
                    for record in successful
                ]
                alloc_peaks = [
                    record["python_alloc_peak_mb"] for record in successful
                ]
                row["wall_time_seconds"] = float(np.median(times))
                row["wall_time_min_seconds"] = float(np.min(times))
                row["wall_time_max_seconds"] = float(np.max(times))
                row["total_wall_time_seconds"] = float(
                    np.median(total_times)
                )
                row["total_wall_time_min_seconds"] = float(
                    np.min(total_times)
                )
                row["total_wall_time_max_seconds"] = float(
                    np.max(total_times)
                )
                row["peak_rss_mb"] = float(np.nanmax(rss_peaks))
                row["peak_rss_delta_mb"] = float(np.nanmax(rss_deltas))
                row["solver_peak_rss_mb"] = float(
                    np.nanmax(solver_rss_peaks)
                )
                row["solver_peak_rss_delta_mb"] = float(
                    np.nanmax(solver_rss_deltas)
                )
                if np.any(np.isfinite(alloc_peaks)):
                    row["python_alloc_peak_mb"] = float(np.nanmax(alloc_peaks))

        if successful_payloads:
            representative_payload = successful_payloads[0]
            summary = representative_payload["summary"]
            for key, value in summary.items():
                row[key] = value

            evaluation = representative_payload["evaluation"]
            if evaluation is not None:
                row.update(
                    {
                        key: value
                        for key, value in evaluation.items()
                    }
                )
                if method == "StaticDO":
                    robust_max = float(
                        evaluation["worst_model_risk"]
                    )
                    row["static_value_consistency_error"] = float(
                        abs(robust_max - row["solver_value"])
                    )

        rows.append(row)

    value_map = {
        row["method"]: row["solver_value"]
        for row in rows
        if row["solver_status"] == "completed"
        and np.isfinite(row["solver_value"])
    }
    hierarchy_ok = np.nan
    if all(method in value_map for method in METHODS):
        hierarchy_ok = bool(
            value_map["NominalDP"]
            <= value_map["StaticDO"] + NUMERICAL_TOLERANCE
            and value_map["StaticDO"]
            <= value_map["HCR"] + NUMERICAL_TOLERANCE
            and value_map["HCR"]
            <= value_map["Rectangular"] + NUMERICAL_TOLERANCE
        )

    static_gap_ok = np.nan
    if "StaticDO" in value_map:
        static_row = next(
            row for row in rows if row["method"] == "StaticDO"
        )
        static_gap_ok = bool(
            np.isfinite(static_row["static_gap"])
            and static_row["static_gap"] <= STATIC_TOLERANCE + 1e-8
        )

    for row in rows:
        row["hierarchy_ok"] = hierarchy_ok
        row["static_gap_ok"] = static_gap_ok

    gc.collect()
    return rows


def run_benchmark():
    configs = build_case_grid()
    all_rows = []
    manifest = {
        "base_seed": BASE_SEED,
        "source_values": list(SOURCE_VALUES),
        "alphabet_values": list(ALPHABET_VALUES),
        "model_values": list(MODEL_VALUES),
        "run_high_source_cases": RUN_HIGH_SOURCE_CASES,
        "run_alphabet_sweep": RUN_ALPHABET_SWEEP,
        "run_model_sweep": RUN_MODEL_SWEEP,
        "max_memo_states": MAX_MEMO_STATES,
        "max_do_iterations": MAX_DO_ITERATIONS,
        "max_case_seconds": MAX_CASE_SECONDS,
        "max_peak_memory_mb": MAX_PEAK_MEMORY_MB,
        "isolate_method_runs": ISOLATE_METHOD_RUNS,
        "multiprocessing_start_method": MP_START_METHOD,
        "memory_measurement_scope": (
            "isolated_child_process"
            if ISOLATE_METHOD_RUNS
            else "long_lived_jupyter_kernel"
        ),
        "process_grace_seconds": PROCESS_GRACE_SECONDS,
        "prior_y1": PRIOR_Y1,
        "abstention_cost": ABSTENTION_COST,
        "static_tolerance": STATIC_TOLERANCE,
        "numerical_tolerance": NUMERICAL_TOLERANCE,
        "cases": configs,
    }
    write_json(OUTPUT_DIR / "benchmark_protocol.json", manifest)

    benchmark_start = time.perf_counter()
    for position, config in enumerate(configs, start=1):
        label = (
            f"{config['sweep']} | m={config['n_sources']} "
            f"A={config['alphabet_size']} K={config['n_models']}"
        )
        print(f"[{position}/{len(configs)}] {label}", flush=True)
        try:
            case_rows = run_one_case(config)
        except Exception as exc:
            case_rows = []
            for method in METHODS:
                row = _empty_method_row(config, method)
                row["status"] = "case_error"
                row["solver_status"] = "case_error"
                row["evaluation_status"] = "not_run"
                row["error"] = "".join(
                    traceback.format_exception_only(type(exc), exc)
                ).strip()
                row["solver_error"] = row["error"]
                case_rows.append(row)
        all_rows.extend(case_rows)
        completed = sum(
            row["solver_status"] == "completed" for row in case_rows
        )
        print(
            f"    solver-completed methods: {completed}/{len(METHODS)}; "
            f"elapsed total: {time.perf_counter() - benchmark_start:.1f}s",
            flush=True,
        )

    frame = pd.DataFrame(all_rows)
    frame.to_csv(OUTPUT_DIR / "solver_scalability_results.csv", index=False)

    # Solver-runtime/state summaries remain usable when a later policy
    # reconstruction/evaluation stage fails. The separate status columns make
    # that distinction explicit instead of discarding a completed solve.
    completed = frame[frame["solver_status"] == "completed"].copy()
    if not completed.empty:
        summary = (
            completed.groupby(
                ["sweep", "n_sources", "alphabet_size", "n_models", "method"],
                as_index=False,
            )[
                [
                    "wall_time_seconds",
                    "total_wall_time_seconds",
                    "peak_rss_mb",
                    "peak_rss_delta_mb",
                    "solver_peak_rss_mb",
                    "solver_peak_rss_delta_mb",
                    "memo_states",
                    "rect_total_beliefs",
                    "rect_max_belief_set",
                    "static_do_iterations",
                    "static_policy_columns",
                    "static_gap",
                    "outcome_count",
                    "reconstruction_seconds",
                    "evaluation_seconds",
                ]
            ]
            .median(numeric_only=True)
        )
    else:
        summary = pd.DataFrame()
    summary.to_csv(OUTPUT_DIR / "solver_scalability_summary.csv", index=False)

    correctness_columns = [
        "sweep",
        "n_sources",
        "alphabet_size",
        "n_models",
        "method",
        "status",
        "solver_status",
        "evaluation_status",
        "hierarchy_ok",
        "static_gap_ok",
        "static_gap",
        "static_value_consistency_error",
    ]
    frame[correctness_columns].to_csv(
        OUTPUT_DIR / "solver_correctness_checks.csv",
        index=False,
    )

    environment = {
        "python": sys.version,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": __import__("scipy").__version__,
        "psutil_available": bool(psutil is not None),
        "platform": platform.platform(),
        "machine": platform.machine(),
        "processor": platform.processor(),
        "cpu_count": os.cpu_count(),
        "blas_threads": {
            key: os.environ.get(key)
            for key in [
                "OMP_NUM_THREADS",
                "MKL_NUM_THREADS",
                "OPENBLAS_NUM_THREADS",
            ]
        },
        "elapsed_seconds": float(time.perf_counter() - benchmark_start),
        "n_cases": int(len(configs)),
        "n_rows": int(len(frame)),
        "isolate_method_runs": bool(ISOLATE_METHOD_RUNS),
        "multiprocessing_start_method": MP_START_METHOD,
        "memory_measurement_scope": (
            "isolated_child_process"
            if ISOLATE_METHOD_RUNS
            else "long_lived_jupyter_kernel"
        ),
    }
    write_json(OUTPUT_DIR / "benchmark_environment.json", environment)

    make_scalability_plots(frame, OUTPUT_DIR)
    return frame

In [ ]:

# Summary plots and final execution



def make_scalability_plots(frame: pd.DataFrame, output_dir: Path):
    try:
        import matplotlib.pyplot as plt
    except Exception as exc:
        warnings.warn(f"Plotting skipped because matplotlib is unavailable: {exc}")
        return

    completed = frame[frame["solver_status"] == "completed"].copy()
    if completed.empty:
        return

    source_data = completed[
        (completed["sweep"] == "source_scaling")
        & (completed["alphabet_size"] == 2)
        & (completed["n_models"] == 3)
    ].copy()
    if not source_data.empty:
        fig, ax = plt.subplots(figsize=(8.5, 5.2))
        for method in METHODS:
            subset = source_data[source_data["method"] == method]
            if not subset.empty:
                ax.plot(
                    subset["n_sources"],
                    subset["wall_time_seconds"],
                    marker="o",
                    label=method,
                )
        ax.set_yscale("log")
        ax.set_xlabel("Number of sources (m)")
        ax.set_ylabel("Solver wall time (seconds, log scale)")
        ax.set_title("ROSE-RAG exact solver scaling with source count")
        ax.grid(True, which="both", alpha=0.25)
        ax.legend()
        fig.tight_layout()
        for suffix in ["png", "pdf", "svg"]:
            fig.savefig(
                output_dir / f"figure_solver_time_vs_sources.{suffix}",
                bbox_inches="tight",
            )
        plt.close(fig)

        fig, ax = plt.subplots(figsize=(8.5, 5.2))
        for method in METHODS:
            subset = source_data[source_data["method"] == method]
            if not subset.empty:
                ax.plot(
                    subset["n_sources"],
                    subset["solver_peak_rss_delta_mb"],
                    marker="o",
                    label=method,
                )
        ax.set_yscale("log")
        ax.set_xlabel("Number of sources (m)")
        ax.set_ylabel("Solver peak RSS increase (MB, log scale)")
        ax.set_title("ROSE-RAG isolated-process solver memory scaling")
        ax.grid(True, which="both", alpha=0.25)
        ax.legend()
        fig.tight_layout()
        for suffix in ["png", "pdf", "svg"]:
            fig.savefig(
                output_dir / f"figure_solver_memory_vs_sources.{suffix}",
                bbox_inches="tight",
            )
        plt.close(fig)

    model_data = completed[
        (completed["sweep"] == "model_scaling")
        & (completed["n_sources"] == 3)
        & (completed["alphabet_size"] == 2)
    ].copy()
    if not model_data.empty:
        fig, ax = plt.subplots(figsize=(8.5, 5.2))
        for method in METHODS:
            subset = model_data[model_data["method"] == method]
            if not subset.empty:
                ax.plot(
                    subset["n_models"],
                    subset["wall_time_seconds"],
                    marker="o",
                    label=method,
                )
        ax.set_yscale("log")
        ax.set_xlabel("Number of complete models (K)")
        ax.set_ylabel("Solver wall time (seconds, log scale)")
        ax.set_title("ROSE-RAG exact solver scaling with model-library size")
        ax.grid(True, which="both", alpha=0.25)
        ax.legend()
        fig.tight_layout()
        for suffix in ["png", "pdf", "svg"]:
            fig.savefig(
                output_dir / f"figure_solver_time_vs_models.{suffix}",
                bbox_inches="tight",
            )
        plt.close(fig)


# RUN_BENCHMARK controls execution after definitions are loaded.
RUN_BENCHMARK = True

if RUN_BENCHMARK:
    results = run_benchmark()
    print("\nBenchmark completed.")
    print("Output directory:", OUTPUT_DIR.resolve())
    display(
        results[
            [
                "sweep",
                "n_sources",
                "alphabet_size",
                "n_models",
                "method",
                "status",
                "solver_status",
                "evaluation_status",
                "wall_time_seconds",
                "solver_peak_rss_delta_mb",
                "memo_states",
                "rect_max_belief_set",
                "static_do_iterations",
                "static_policy_columns",
                "hierarchy_ok",
            ]
        ].sort_values(
            ["sweep", "n_sources", "alphabet_size", "n_models", "method"]
        )
    )

## Output files

The output directory contains per-run results, summaries, correctness checks, protocol and environment records. Solver and evaluation statuses are recorded separately. CSV/JSON records provide the numerical basis for tables; figures are derived views.
